## Step 7 — create candidate new blocks from smoothed building clusters
**# of cells in notebook:** 1

**Purpose:** Convert the smoothed building-cluster classifications from Step 6 into candidate new block polygons. For each available clustering solution, building-level tessellation cells assigned to the same smoothed cluster are dissolved to create candidate subdivisions of the original block.

**Input:**

- `building_level_tessellation.gpkg` from Step 3
- `scmc_new_dist_relabel.gdb` from Step 6
- `context_bldg_id` in the tessellation
- `context_bldg_id` in the smoothed building-cluster layers; `SOURCE_ID` is accepted as a fallback for legacy outputs
- `cluster_smooth`, `cell_area_m2`, and `area_m_utm` carried by the Step 6 building layers

**Output:**

Within each block folder:

- `new_blocks.gpkg`

with available layers:

- `blk_1_<source_block>_2`
- `blk_1_<source_block>_3`
- `blk_1_<source_block>_4`
- `blk_1_<source_block>_5`

Each layer represents one candidate subdivision of the original block.

**Main logic:**

**Cell 1 — Dissolve tessellation cells by smoothed cluster**

1. Reads the Step 3 building-level tessellation and the available smoothed clustering layers from Step 6.
2. Joins each tessellation cell to its corresponding building using `context_bldg_id`; `SOURCE_ID` is used only as a fallback for legacy Step 6 outputs.
3. Transfers `cluster_smooth`, `cell_area_m2`, `area_m_utm`, and `block_folder` from the Step 6 building layer to the tessellation cells.
4. Excludes tessellation cells that cannot be matched to a smoothed building record.
5. Dissolves tessellation cells by `cluster_smooth` for each available value of k.
6. Sums tessellation-cell area and building area for each candidate new block and adds fields identifying the source block and value of k.
7. Writes one candidate block layer per value of k to `new_blocks.gpkg`.


In [ ]:
# -*- coding: utf-8 -*-
r"""
Create dissolved tessellation-block polygons from smoothed SCMC building clusters
WITHOUT ArcPy.

For each block folder under:
    E:\_johannesburg\_analysis\heterogeneous_largePop_blocks\_<block_id>

the script reads:
    building_level_tessellation.gpkg / building_level_tessellation
    scmc_new_dist_relabel.gdb / scmc_k2_relabel ... scmc_k5_relabel

For each k, it joins the Step 3 tessellation cells to the corresponding
Step 6 building records. The preferred join is:

    scmc context_bldg_id == tessellation context_bldg_id

For compatibility with older outputs, SOURCE_ID is accepted as a fallback
when context_bldg_id is not present in the Step 6 layer.

The Step 6 layer supplies cluster_smooth, cell_area_m2, area_m_utm, and
block_folder. Tessellation geometry is then dissolved by cluster_smooth.

Outputs:
    <block_folder>\new_blocks.gpkg

Output layers:
    blk_1_<source_block>_<k>

Output fields:
    cluster_smooth
    cell_area_m2      = sum of Step 6 building cell_area_m2
    area_m_utm        = sum of Step 6 building area_m_utm
    block_folder      = first block_folder value
    source_block      = number after underscore in block folder
    k_clusters        = k
    block_id          = blk_1_<source_block>_<k>

Notes:
    - This script uses GeoPandas / pyogrio / pandas only.
    - It does not use arcpy.
    - It writes outputs to GeoPackage, not File Geodatabase.
    - large_pop_orig is intentionally NOT assigned here. Step 8 looks up the
      original screening flags and writes the correct value.
"""

import re
import traceback
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyogrio


# ---------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------

BASE_DIR = Path(r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks")

K_VALUES = [2, 3, 4, 5]

TESS_GPKG_NAME = "building_level_tessellation.gpkg"
TESS_LAYER_PREFERRED = "building_level_tessellation"

SCMC_GDB_NAME = "scmc_new_dist_relabel.gdb"
SCMC_LAYER_TEMPLATE = "scmc_k{k}_relabel"

OUT_GPKG_NAME = "new_blocks.gpkg"

TESS_ID_FIELD = "context_bldg_id"

# Revised Steps 3–6 preserve context_bldg_id all the way into the smoothed
# building layer. SOURCE_ID is retained as a fallback for older SCMC outputs.
SCMC_ID_FIELD_PREFERRED = "context_bldg_id"
SCMC_ID_FIELD_FALLBACK = "SOURCE_ID"

CLUSTER_FIELD = "cluster_smooth"
SUM_FIELDS = ["cell_area_m2", "area_m_utm"]
FIRST_FIELDS = ["block_folder"]

OVERWRITE_OUTPUTS = True

# If True, disconnected polygons with the same cluster_smooth value become one
# multipart feature. If False, multipart outputs are exploded into separate rows.
CREATE_MULTIPART_DISSOLVES = True

# Optional. Sometimes invalid geometries can slow or break dissolve operations.
FIX_INVALID_GEOMETRIES = False


# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def msg(text=""):
    print(text, flush=True)


def normalize_join_key(value):
    """Normalize numeric/string building IDs to a consistent string key."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        f = float(value)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    s = str(value).strip()
    if not s:
        return None

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def list_layer_names(dataset_path):
    """Return layer names from a GeoPackage or File Geodatabase."""
    layers = pyogrio.list_layers(str(dataset_path))
    if hasattr(layers, "shape"):
        return [str(row[0]) for row in layers]
    return [str(row[0]) if isinstance(row, (list, tuple)) else str(row) for row in layers]


def find_layer(dataset_path, preferred_name, fallback_suffix):
    """Find a layer by exact preferred name, then by suffix."""
    if not Path(dataset_path).exists():
        raise RuntimeError(f"Dataset not found: {dataset_path}")

    layers = list_layer_names(dataset_path)
    if not layers:
        raise RuntimeError(f"No layers found inside: {dataset_path}")

    preferred_lower = preferred_name.lower()
    suffix_lower = fallback_suffix.lower()

    for layer in layers:
        if layer.lower() == preferred_lower:
            return layer

    for layer in layers:
        if layer.lower().endswith(suffix_lower):
            return layer

    raise RuntimeError(
        f"Could not find layer in {dataset_path}. "
        f"Preferred='{preferred_name}', suffix='{fallback_suffix}'. "
        f"Found layers={layers}"
    )


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]
    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def actual_column_name(df, requested_name):
    requested_lower = requested_name.lower()
    for col in df.columns:
        if col.lower() == requested_lower:
            return col
    raise RuntimeError(f"Column not found: {requested_name}")


def first_existing_column(df, preferred, fallback=None):
    """Return preferred column if present; otherwise optional fallback."""
    lower_lookup = {c.lower(): c for c in df.columns}

    if preferred.lower() in lower_lookup:
        return lower_lookup[preferred.lower()]

    if fallback is not None and fallback.lower() in lower_lookup:
        return lower_lookup[fallback.lower()]

    if fallback is None:
        raise RuntimeError(f"Column not found: {preferred}")

    raise RuntimeError(
        f"Neither preferred nor fallback ID field was found: "
        f"{preferred}, {fallback}"
    )


def iter_block_folders(base_dir):
    """Yield (folder_path, source_block) for directories named _<digits>."""
    for item in sorted(base_dir.iterdir()):
        if not item.is_dir():
            continue

        m = re.fullmatch(r"_(\d+)", item.name)
        if not m:
            continue

        yield item, m.group(1)


def read_tessellation(tess_gpkg):
    """Read the Step 3 building-level tessellation."""
    tess_layer = find_layer(
        dataset_path=tess_gpkg,
        preferred_name=TESS_LAYER_PREFERRED,
        fallback_suffix="building_level_tessellation",
    )

    msg(f"  Tessellation layer: {tess_layer}")

    tess = gpd.read_file(str(tess_gpkg), layer=tess_layer)
    require_columns(tess, [TESS_ID_FIELD], "building_level_tessellation")

    return tess


def read_scmc_table(scmc_gdb, scmc_layer_name):
    """
    Read Step 6 attributes needed to dissolve tessellation cells.

    The revised workflow prefers context_bldg_id because it is the stable
    identifier created from the original block-building FileGDB feature ID.
    SOURCE_ID remains a legacy fallback.
    """
    layers = list_layer_names(scmc_gdb)
    lower_to_actual = {layer.lower(): layer for layer in layers}

    if scmc_layer_name.lower() not in lower_to_actual:
        return None

    actual_layer = lower_to_actual[scmc_layer_name.lower()]

    scmc = pyogrio.read_dataframe(
        str(scmc_gdb),
        layer=actual_layer,
        read_geometry=False,
    )

    id_col = first_existing_column(
        scmc,
        SCMC_ID_FIELD_PREFERRED,
        SCMC_ID_FIELD_FALLBACK,
    )

    require_columns(
        scmc,
        [CLUSTER_FIELD] + SUM_FIELDS + FIRST_FIELDS,
        scmc_layer_name,
    )

    return scmc, id_col


def make_block_output(tess, scmc, scmc_id_col, source_block, k):
    """Create one dissolved candidate-block GeoDataFrame for one k value."""
    tess_id_col = actual_column_name(tess, TESS_ID_FIELD)
    cluster_col = actual_column_name(scmc, CLUSTER_FIELD)
    sum_cols = [actual_column_name(scmc, c) for c in SUM_FIELDS]
    first_cols = [actual_column_name(scmc, c) for c in FIRST_FIELDS]

    lookup_cols = [scmc_id_col, cluster_col] + sum_cols + first_cols
    lookup = scmc[lookup_cols].copy()
    lookup["_join_key"] = lookup[scmc_id_col].apply(normalize_join_key)

    before = len(lookup)
    lookup = lookup[lookup["_join_key"].notna()].copy()
    lookup = lookup.drop_duplicates(subset=["_join_key"], keep="last")
    duplicate_count = before - len(lookup)

    if duplicate_count:
        msg(
            f"    Warning: {duplicate_count:,} duplicate/blank building ID "
            "value(s) dropped; last value kept."
        )

    rename_map = {cluster_col: CLUSTER_FIELD}
    rename_map.update({actual: requested for requested, actual in zip(SUM_FIELDS, sum_cols)})
    rename_map.update({actual: requested for requested, actual in zip(FIRST_FIELDS, first_cols)})

    lookup = lookup.rename(columns=rename_map)
    lookup = lookup[["_join_key", CLUSTER_FIELD] + SUM_FIELDS + FIRST_FIELDS]

    work = tess.copy()
    work["_join_key"] = work[tess_id_col].apply(normalize_join_key)

    joined = work.merge(
        lookup,
        on="_join_key",
        how="left",
        validate="many_to_one",
    )

    matched = int(joined[CLUSTER_FIELD].notna().sum())
    unmatched = int(joined[CLUSTER_FIELD].isna().sum())

    msg(f"    Matched tessellation cells: {matched:,}")
    msg(f"    Unmatched tessellation cells: {unmatched:,}")
    msg(f"    Step 6 building IDs:        {len(lookup):,}")
    msg(f"    Step 6 ID field used:       {scmc_id_col}")

    if matched == 0:
        raise RuntimeError("No tessellation cells matched Step 6 building IDs.")

    # Exclude unmatched cells. These correspond to source buildings that did not
    # receive a Step 6 record or, for rare tiny slivers, did not receive a Step 3
    # tessellation cell.
    joined = joined[joined[CLUSTER_FIELD].notna()].copy()

    if FIX_INVALID_GEOMETRIES:
        joined["geometry"] = joined.geometry.make_valid()

    keep_cols = [CLUSTER_FIELD] + SUM_FIELDS + FIRST_FIELDS + ["geometry"]
    joined = joined[keep_cols].copy()

    dissolved = joined.dissolve(
        by=CLUSTER_FIELD,
        aggfunc={
            "cell_area_m2": "sum",
            "area_m_utm": "sum",
            "block_folder": "first",
        },
        as_index=False,
        sort=True,
    )

    if not CREATE_MULTIPART_DISSOLVES:
        dissolved = dissolved.explode(index_parts=False, ignore_index=True)

    block_id = f"blk_1_{source_block}_{k}"

    dissolved["source_block"] = int(source_block)
    dissolved["k_clusters"] = int(k)
    dissolved["block_id"] = block_id

    dissolved = dissolved[
        [
            CLUSTER_FIELD,
            "cell_area_m2",
            "area_m_utm",
            "block_folder",
            "source_block",
            "k_clusters",
            "block_id",
            "geometry",
        ]
    ].copy()

    return dissolved


def write_layer(gdf, out_gpkg, layer_name):
    """Write one layer to the block's output GeoPackage."""
    gdf.to_file(
        str(out_gpkg),
        layer=layer_name,
        driver="GPKG",
        engine="pyogrio",
    )


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------

def main():
    msg("Starting dissolved tessellation block creation")
    msg("Using GeoPandas / pyogrio only. No ArcPy.")
    msg(f"Base directory: {BASE_DIR}")
    msg("")

    if not BASE_DIR.exists():
        raise RuntimeError(f"BASE_DIR does not exist: {BASE_DIR}")

    processed = 0
    skipped = 0
    failed = 0
    output_layers_written = 0

    for block_folder, source_block in iter_block_folders(BASE_DIR):
        processed += 1

        msg("=" * 80)
        msg(f"Block folder: {block_folder}")
        msg(f"source_block: {source_block}")

        try:
            tess_gpkg = block_folder / TESS_GPKG_NAME
            scmc_gdb = block_folder / SCMC_GDB_NAME
            out_gpkg = block_folder / OUT_GPKG_NAME

            if not tess_gpkg.exists():
                skipped += 1
                msg(f"  Skipping: missing {TESS_GPKG_NAME}")
                continue

            if not scmc_gdb.exists():
                skipped += 1
                msg(f"  Skipping: missing {SCMC_GDB_NAME}")
                continue

            if OVERWRITE_OUTPUTS and out_gpkg.exists():
                msg(f"  Removing existing output GeoPackage: {out_gpkg}")
                out_gpkg.unlink()

            tess = read_tessellation(tess_gpkg)

            for k in K_VALUES:
                scmc_layer_name = SCMC_LAYER_TEMPLATE.format(k=k)
                out_layer_name = f"blk_1_{source_block}_{k}"

                msg("")
                msg(f"  k={k}")
                msg(f"    SCMC layer: {scmc_gdb} | {scmc_layer_name}")
                msg(f"    Output:     {out_gpkg} | {out_layer_name}")

                scmc_result = read_scmc_table(scmc_gdb, scmc_layer_name)

                if scmc_result is None:
                    msg(f"    Skipping k={k}: layer not found.")
                    continue

                scmc, scmc_id_col = scmc_result

                dissolved = make_block_output(
                    tess=tess,
                    scmc=scmc,
                    scmc_id_col=scmc_id_col,
                    source_block=source_block,
                    k=k,
                )

                write_layer(
                    gdf=dissolved,
                    out_gpkg=out_gpkg,
                    layer_name=out_layer_name,
                )

                msg(f"    Wrote {len(dissolved):,} dissolved cluster feature(s).")
                output_layers_written += 1

        except Exception:
            failed += 1
            msg("  FAILED on this block:")
            msg(traceback.format_exc())

    msg("")
    msg("=" * 80)
    msg("Finished.")
    msg(f"Block folders visited:       {processed}")
    msg(f"Block folders skipped:       {skipped}")
    msg(f"Block folders failed:        {failed}")
    msg(f"Output layers written:       {output_layers_written}")


if __name__ == "__main__":
    main()
